# Pair-bias structural pattern analysis

Loads a capture file produced by `capture_tensors.py` and computes four metrics:

1. **Symmetry** — fraction of squared energy in the symmetric part.
2. **Spatial sparsity** — fraction of 64×64 tiles with mean |B| below 10% of median.
3. **Temporal reuse** — `||B_{k+1} − B_k|| / ||B_k||` between consecutive Pairformer blocks.
4. **Cross-head correlation** — average pairwise Pearson correlation between heads.

Each cell plots and prints a summary; the final cell renders the decision table.


In [ ]:
import re
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt

CAPTURE_PATH = 'captures/run1.pt'  # change me

blob = torch.load(CAPTURE_PATH, map_location='cpu')
if isinstance(blob, dict) and 'captures' in blob:
    meta = blob.get('_meta', {})
    data = blob['captures']
else:  # backwards-compatible: plain dict of tensors
    meta, data = {}, blob

print(f'Loaded {len(data)} capture points')
if meta:
    print('meta:', meta)
if meta.get('weights_initialized') is False:
    print('\nWARNING: weights look un-trained; measured patterns are noise, not signal.')


## Organize captures by block index + layer type

In [ ]:
def parse_name(name: str):
    m = re.search(r'blocks\.(\d+)\.(\w+)', name)
    if m:
        return int(m.group(1)), m.group(2)
    return None, None

by_layer: dict[str, dict[int, torch.Tensor]] = defaultdict(dict)
for name, tensor in data.items():
    idx, ltype = parse_name(name)
    if idx is None:
        continue
    by_layer[ltype][idx] = tensor

for ltype, blocks in by_layer.items():
    example = next(iter(blocks.values()))
    print(f'  {ltype}: {len(blocks)} blocks, shape {tuple(example.shape)}, dtype {example.dtype}')

LAYER_TYPES = [
    'attention_pair_bias',
    'triangle_attention_starting',
    'triangle_attention_ending',
]


## Property 1 — Symmetry

`symmetry_ratio(B) = ||B+B.T||^2 / (||B+B.T||^2 + ||B-B.T||^2)` per head.
Expectation: near 1.0 indicates strong symmetry; 0.5 means none.

In [ ]:
def _as_hnn(B: torch.Tensor) -> torch.Tensor:
    '''Normalize any pair_bias tensor to [H, N, N] for a single batch item.'''
    B = B.float()
    if B.dim() == 5:   # [B, 1, H, N, N]
        B = B[0, 0]
    elif B.dim() == 4: # [B, H, N, N]
        B = B[0]
    else:
        raise ValueError(f'Unexpected shape {tuple(B.shape)}')
    return B

def symmetry_ratio(B: torch.Tensor) -> float:
    B = _as_hnn(B)
    sym = 0.5 * (B + B.transpose(-2, -1))
    anti = 0.5 * (B - B.transpose(-2, -1))
    se = (sym ** 2).sum().item()
    ae = (anti ** 2).sum().item()
    return se / (se + ae + 1e-12)

fig, ax = plt.subplots(figsize=(10, 4))
sym_summary = {}
for ltype in LAYER_TYPES:
    if ltype not in by_layer:
        continue
    blocks = sorted(by_layer[ltype])
    values = [symmetry_ratio(by_layer[ltype][b]) for b in blocks]
    ax.plot(blocks, values, 'o-', label=ltype, markersize=4)
    sym_summary[ltype] = values
    print(f'{ltype}: mean={np.mean(values):.3f}, min={min(values):.3f}, max={max(values):.3f}')

ax.axhline(0.5, color='gray', ls='--', alpha=0.5, label='no symmetry')
ax.axhline(0.9, color='red', ls='--', alpha=0.5, label='strong symmetry')
ax.set_xlabel('Pairformer block index')
ax.set_ylabel('symmetric energy ratio')
ax.set_ylim(0, 1.02)
ax.set_title('Property 1: symmetry of pair_bias across blocks')
ax.legend()
plt.tight_layout()
plt.show()


## Property 2 — Spatial sparsity (tile-level)

64×64 tiles, "low" = mean |value| below 10% of the per-head median.

In [ ]:
def tile_magnitude_map(B: torch.Tensor, tile: int = 64) -> torch.Tensor:
    B = _as_hnn(B)
    H, N, _ = B.shape
    ntiles = N // tile
    if ntiles == 0:
        return B.abs().mean(dim=(-2, -1), keepdim=True)
    Nc = ntiles * tile
    B = B[:, :Nc, :Nc]
    tiles = B.reshape(H, ntiles, tile, ntiles, tile)
    return tiles.abs().mean(dim=(-3, -1))

# Visual sanity check on a middle block (or last available)
vis_ltype = 'attention_pair_bias' if 'attention_pair_bias' in by_layer else LAYER_TYPES[0]
if vis_ltype in by_layer:
    blocks = sorted(by_layer[vis_ltype])
    vis_idx = blocks[len(blocks) // 2]
    tm = tile_magnitude_map(by_layer[vis_ltype][vis_idx], tile=64)
    H = tm.shape[0]
    fig, axes = plt.subplots(1, min(H, 4), figsize=(3 * min(H, 4), 3))
    if min(H, 4) == 1:
        axes = [axes]
    for h in range(min(H, 4)):
        im = axes[h].imshow(tm[h].numpy(), cmap='viridis')
        axes[h].set_title(f'head {h}')
        plt.colorbar(im, ax=axes[h])
    plt.suptitle(f'Per-tile |pair_bias|  —  {vis_ltype}, block {vis_idx}, tile=64')
    plt.tight_layout()
    plt.show()

# Aggregate
sparsity_summary = {}
for ltype in LAYER_TYPES:
    if ltype not in by_layer:
        continue
    fracs = []
    for B in by_layer[ltype].values():
        tm = tile_magnitude_map(B, tile=64)
        thresh = tm.median() * 0.1
        fracs.append((tm < thresh).float().mean().item())
    sparsity_summary[ltype] = fracs
    print(f'{ltype}: mean low-tile frac = {np.mean(fracs):.2%}  range {min(fracs):.2%} .. {max(fracs):.2%}')


## Property 3 — Temporal reuse between consecutive Pairformer blocks

In [ ]:
def temporal_deltas(block_dict: dict[int, torch.Tensor]):
    idx = sorted(block_dict)
    out = []
    for k in range(len(idx) - 1):
        a = _as_hnn(block_dict[idx[k]])
        b = _as_hnn(block_dict[idx[k + 1]])
        if a.shape != b.shape:
            out.append((idx[k], None))
            continue
        out.append((idx[k], ((b - a).norm() / (a.norm() + 1e-12)).item()))
    return out

fig, ax = plt.subplots(figsize=(10, 4))
temporal_summary = {}
for ltype in LAYER_TYPES:
    if ltype not in by_layer:
        continue
    deltas = temporal_deltas(by_layer[ltype])
    xs = [x for x, d in deltas if d is not None]
    ys = [d for x, d in deltas if d is not None]
    if not ys:
        continue
    ax.plot(xs, ys, 'o-', label=ltype, markersize=4)
    temporal_summary[ltype] = ys
    print(f'{ltype}: mean delta = {np.mean(ys):.3f}  median = {np.median(ys):.3f}')

ax.axhline(0.1, color='red', ls='--', alpha=0.5, label='strong reuse (<0.1)')
ax.axhline(0.3, color='orange', ls='--', alpha=0.5, label='weak reuse (>0.3)')
ax.set_yscale('log')
ax.set_xlabel('block index k')
ax.set_ylabel(r'$\|B_{k+1}-B_k\| / \|B_k\|$')
ax.set_title('Property 3: temporal change across Pairformer blocks')
ax.legend()
plt.tight_layout()
plt.show()


## Property 4 — Cross-head correlation

In [ ]:
def head_correlation(B: torch.Tensor) -> torch.Tensor:
    B = _as_hnn(B)
    H = B.shape[0]
    flat = B.reshape(H, -1)
    flat = flat - flat.mean(dim=1, keepdim=True)
    norms = flat.norm(dim=1, keepdim=True)
    normed = flat / (norms + 1e-12)
    return normed @ normed.T

crosshead_summary = {}
for ltype in LAYER_TYPES:
    if ltype not in by_layer:
        continue
    corrs = [head_correlation(B) for B in by_layer[ltype].values()]
    avg = torch.stack(corrs).mean(dim=0)
    H = avg.shape[0]
    mask = 1.0 - torch.eye(H)
    mean_off = ((avg * mask).sum() / mask.sum()).item()
    crosshead_summary[ltype] = mean_off
    fig, ax = plt.subplots(figsize=(4.5, 3.6))
    im = ax.imshow(avg.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_title(f'{ltype}\navg cross-head corr   mean off-diag = {mean_off:.3f}')
    ax.set_xlabel('head'); ax.set_ylabel('head')
    plt.tight_layout()
    plt.show()
    print(f'{ltype}: mean off-diagonal correlation = {mean_off:.3f}')


## Summary and decision

In [ ]:
def strength(val, strong, moderate, direction='high'):
    if direction == 'high':
        if val >= strong:   return 'STRONG'
        if val >= moderate: return 'MODERATE'
        return 'WEAK'
    else:  # lower is better
        if val <= strong:   return 'STRONG'
        if val <= moderate: return 'MODERATE'
        return 'WEAK'

print('=' * 72)
print('STRUCTURAL PROPERTY SUMMARY')
print('=' * 72)

for ltype in LAYER_TYPES:
    if ltype not in by_layer:
        continue
    print(f'\n[{ltype}]')

    if ltype in sym_summary:
        v = float(np.mean(sym_summary[ltype]))
        print(f'  1. Symmetry                 mean = {v:.3f}   (threshold 0.9)     {strength(v, 0.9, 0.7)}')

    if ltype in sparsity_summary:
        v = float(np.mean(sparsity_summary[ltype]))
        print(f'  2. Spatial sparsity         mean = {v:.2%}   (threshold 30%)     {strength(v, 0.30, 0.15)}')

    if ltype in temporal_summary:
        v = float(np.median(temporal_summary[ltype]))
        print(f'  3. Temporal reuse           med  = {v:.3f}   (threshold 0.1)     {strength(v, 0.1, 0.3, direction="low")}')

    if ltype in crosshead_summary:
        v = float(crosshead_summary[ltype])
        print(f'  4. Cross-head correlation   mean = {v:.3f}   (threshold 0.8)     {strength(v, 0.8, 0.5)}')
